In [ ]:
# Unified analysis function
from pathlib import Path
import re
import csv
import yaml
import numpy as np
import torch
from scipy.signal import find_peaks
import matplotlib.pyplot as plt

from infer_beat import (
    build_model,
    apply_position_mode,
    normalize_feats,
    build_beat_windows,
    slice_by_beats,
    apply_pos_weight_calibration,
)


def analyze_boundary_dist(
    config_path,
    ckpt_path,
    input_dir,
    pattern,
    pos_weight=None,
    pred_height=0.05,
    pred_min_dist=6,
    pred_prominence=0.03,
    use_true_binary=True,
    true_height=0.5,
    true_min_dist=6,
    true_prominence=0.05,
    assign_radius=6,
    output_csv=None,
    do_plots=True,
):
    """
    End-to-end analysis for dist models.
    Inputs:
      - config_path: YAML config path
      - ckpt_path: model .pt path
      - input_dir: directory containing npz files
      - pattern: glob pattern (e.g., "M06-3_*_L6.npz")
      - pos_weight: optional calibration weight for logits (only used if loss_type==bce)
    """
    cfg_path = Path(config_path)
    ckpt_path = Path(ckpt_path)
    input_dir = Path(input_dir)

    cfg = yaml.safe_load(cfg_path.read_text())

    # auto-adjust layers from checkpoint
    state = torch.load(ckpt_path, map_location="cpu")
    max_idx = -1
    for k in state.keys():
        m = re.match(r"beat_encoder\\.layers\\.(\\d+)\\.", k)
        if m:
            max_idx = max(max_idx, int(m.group(1)))
    if max_idx >= 0:
        cfg["model"]["num_layers"] = max_idx + 1

    # performer conditioning if present in checkpoint
    has_perf = any(k.startswith("performer_") for k in state.keys())
    if has_perf:
        cfg.setdefault("model", {})["performer_cond"] = True
        cfg["model"]["performer_vocab_size"] = int(state["performer_emb.weight"].shape[0])
        cfg["model"]["performer_emb_dim"] = int(state["performer_emb.weight"].shape[1])
    else:
        cfg.setdefault("model", {})["performer_cond"] = False

    files = sorted(input_dir.glob(pattern))
    if not files:
        raise RuntimeError(f"No matching npz files: {input_dir}/{pattern}")

    sample = np.load(files[0])
    input_dim = sample["note_feats"].shape[1]

    model = build_model(cfg, input_dim=input_dim)
    model.load_state_dict(state, strict=True)
    model.eval()

    position_mode = cfg.get("data", {}).get("position_mode", "absolute")
    value_ranges = cfg.get("data", {}).get("value_ranges")
    window_beats = cfg.get("data", {}).get("beat_sequence_length")
    window_stride = cfg.get("data", {}).get("beat_stride") or window_beats
    max_len = cfg.get("data", {}).get("max_len")
    include_empty_beats = cfg.get("model", {}).get("include_empty_beats", False)
    loss_type = cfg.get("training", {}).get("loss_type", "bce")

    def infer_probs_for_npz(npz_path):
        npz = np.load(npz_path)
        note_feats = npz["note_feats"]
        beat_ids = npz["beat_ids"]
        if beat_ids.size == 0:
            return np.array([])

        num_beats = int(beat_ids.max()) + 1

        if window_beats is None:
            feats = apply_position_mode(note_feats, position_mode)
            if value_ranges:
                feats = normalize_feats(feats, value_ranges)
            feats_t = torch.tensor(feats).unsqueeze(0)
            beat_ids_t = torch.tensor(beat_ids).unsqueeze(0)
            attn_mask = beat_ids_t >= 0
            with torch.no_grad():
                logits, _ = model(
                    feats_t,
                    beat_ids=beat_ids_t,
                    num_beats=num_beats,
                    attn_mask=attn_mask,
                    labels=None,
                    output_head="dist",
                )
                if pos_weight is not None and loss_type == "bce":
                    logits = apply_pos_weight_calibration(logits, float(pos_weight))
                probs = torch.sigmoid(logits).squeeze(0).cpu().numpy()
            if not include_empty_beats:
                valid_beats = np.zeros(num_beats, dtype=bool)
                valid_beats[np.unique(beat_ids[beat_ids >= 0])] = True
                probs = np.where(valid_beats, probs, 0.0)
            return probs

        windows = build_beat_windows(num_beats, int(window_beats), int(window_stride))
        sum_probs = np.zeros(num_beats, dtype=np.float64)
        count_probs = np.zeros(num_beats, dtype=np.int64)

        for start, end in windows:
            feats, ids = slice_by_beats(note_feats, beat_ids, start, end, max_len)
            if feats is None or ids is None or feats.shape[0] == 0:
                continue
            feats = apply_position_mode(feats, position_mode)
            if value_ranges:
                feats = normalize_feats(feats, value_ranges)
            feats_t = torch.tensor(feats).unsqueeze(0)
            beat_ids_t = torch.tensor(ids).unsqueeze(0)
            attn_mask = beat_ids_t >= 0
            with torch.no_grad():
                logits, _ = model(
                    feats_t,
                    beat_ids=beat_ids_t,
                    num_beats=end - start,
                    attn_mask=attn_mask,
                    labels=None,
                    output_head="dist",
                )
                if pos_weight is not None and loss_type == "bce":
                    logits = apply_pos_weight_calibration(logits, float(pos_weight))
                probs_win = torch.sigmoid(logits).squeeze(0).cpu().numpy()

            idx = np.arange(start, end)
            if include_empty_beats:
                sum_probs[idx] += probs_win
                count_probs[idx] += 1
            else:
                valid = np.zeros(end - start, dtype=bool)
                if ids.size > 0:
                    valid[np.unique(ids[ids >= 0])] = True
                sum_probs[idx[valid]] += probs_win[valid]
                count_probs[idx[valid]] += 1

        probs = np.zeros(num_beats, dtype=np.float64)
        seen = count_probs > 0
        probs[seen] = sum_probs[seen] / count_probs[seen]
        return probs.astype(np.float32)

    pred_probs = infer_probs_for_npz(files[0])
    pred_peaks, _ = find_peaks(
        pred_probs,
        distance=pred_min_dist,
        height=pred_height,
        prominence=pred_prominence,
    )

    num_beats = len(np.load(files[0])["boundary_probs"])
    true_counts = np.zeros(num_beats, dtype=int)

    for p in files:
        labels = np.load(p)["boundary_probs"].astype(float)
        if use_true_binary:
            true_peaks = np.where(labels > 0.5)[0]
        else:
            true_peaks, _ = find_peaks(
                labels,
                distance=true_min_dist,
                height=true_height,
                prominence=true_prominence,
            )
        true_counts[true_peaks] += 1

    true_prob = true_counts / len(files)

    true_avg_peaks = true_counts.sum() / max(len(files), 1)
    pred_peak_count = int(len(pred_peaks))

    # align pred peaks to nearest true peaks
    true_peaks = np.where(true_counts > 0)[0]
    assignments = {int(t): [] for t in true_peaks}
    for pk in pred_peaks:
        if true_peaks.size == 0:
            continue
        nearest = int(true_peaks[np.argmin(np.abs(true_peaks - pk))])
        if abs(nearest - pk) <= assign_radius:
            assignments[nearest].append((int(pk), float(pred_probs[pk])))

    # compute relative strengths
    rows = []
    pred_sum_total = 0.0
    for t in true_peaks:
        assigned = assignments[int(t)]
        if assigned:
            strengths = np.array([s for _, s in assigned], dtype=float)
            pred_sum = float(strengths.sum())
            pred_max = float(strengths.max())
            pred_cnt = len(strengths)
            pred_list = " ".join(str(p) for p, _ in assigned)
        else:
            pred_sum = 0.0
            pred_max = 0.0
            pred_cnt = 0
            pred_list = ""
        pred_sum_total += pred_sum
        rows.append([int(t), int(true_counts[int(t)]), pred_cnt, pred_list, pred_sum, pred_max])

    true_sum_total = float(true_counts[true_peaks].sum()) if true_peaks.size > 0 else 0.0
    rows_rel = []
    for t, true_cnt, pred_cnt, pred_list, pred_sum, pred_max in rows:
        true_rel = (true_cnt / true_sum_total) if true_sum_total > 0 else 0.0
        pred_rel = (pred_sum / pred_sum_total) if pred_sum_total > 0 else 0.0
        rows_rel.append([t, true_cnt, true_rel, pred_cnt, pred_list, pred_sum, pred_rel, pred_max])

    if output_csv is None:
        output_csv = input_dir / "pred_vs_true.csv"
    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    with output_csv.open("w", newline="") as f:
        w = csv.writer(f)
        w.writerow([
            "true_peak", "true_count", "true_rel",
            "pred_count", "pred_peaks", "pred_sum", "pred_rel", "pred_max"
        ])
        w.writerows(rows_rel)

    print(f"true_avg_peaks_per_perf: {true_avg_peaks:.4f}")
    print(f"pred_peak_count: {pred_peak_count}")

    if do_plots:
        tp = np.array([r[0] for r in rows_rel], dtype=int)
        true_rel = np.array([r[2] for r in rows_rel], dtype=float)
        pred_rel = np.array([r[6] for r in rows_rel], dtype=float)

        plt.figure(figsize=(12, 5))
        plt.plot(pred_probs, label="Pred curve (dist)", linewidth=1.2, color="C0")
        plt.plot(tp, true_rel, label="True relative strength", linewidth=1.2, color="C1")
        plt.title("Pred Curve vs True Relative Strength")
        plt.xlabel("Beat index")
        plt.ylabel("Value")
        plt.legend()
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(12, 6))
        plt.bar(np.arange(num_beats), true_prob, width=1.0, alpha=0.6, label="True peak frequency")
        for i, pk in enumerate(pred_peaks):
            plt.axvline(pk, color="red", linewidth=1.0, alpha=0.8, label="Pred peaks" if i == 0 else None)
        plt.title("True peak distribution vs Pred peaks")
        plt.xlabel("Beat index")
        plt.ylabel("True peak frequency (across performers)")
        plt.legend()
        plt.tight_layout()
        plt.show()

    return {
        "true_avg_peaks": float(true_avg_peaks),
        "pred_peak_count": int(pred_peak_count),
        "pred_peaks": pred_peaks,
        "true_peaks": true_peaks,
        "output_csv": output_csv,
    }


# Example call (edit paths as needed)
# result = analyze_boundary_dist(
#     config_path="config_beat_mazurka_level6.yaml",
#     ckpt_path="check/beat_mazurka/level_6/beat_mazurka_L6_20260130_141117/best.pt",
#     input_dir="beat_data_mazurka_performer_levels",
#     pattern="M06-3_*_L6.npz",
#     pos_weight=None,
# )
